In [84]:
import os
import glob
import shutil
import random

import numpy as np
import xml.etree.ElementTree as ET
import yaml
from tqdm.contrib.concurrent import thread_map

In [85]:
BASE_FOLDERS = [
    "/home/kyamkovyi/Pictures/cam4_poc_part1",
    "/home/kyamkovyi/Pictures/4pplwalking",
]

In [86]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)


def create_label_txt(args) -> None:
    pth, obj = args
    with open(os.path.join(pth, f"{obj["id"]}.txt"), "wt") as f:
        for det in obj["objects"]:
            w = det["xbr"] - det["xtl"]
            h = det["ybr"] - det["ytl"]
            x_c = det["xtl"] + w / 2
            y_c = det["ytl"] + h / 2
            f.write(f"{det['cls']} {x_c} {y_c} {w} {h} {det['x']} {det['y']}\n")

def cp_img(args) -> None:
    pth, obj = args
    shutil.copy(
        os.path.join(obj["path"]),
        os.path.join(pth, f"{obj['id']}{os.path.splitext(obj['path'])[-1]}")
    )


def split(objects: list, val_size: float, random_state: int) -> tuple:
    n = len(objects)
    n_val = int(n * val_size)
    idxs = np.arange(n)
    np.random.seed(random_state)
    np.random.shuffle(idxs)
    train_idxs = idxs[:-n_val]
    val_idxs = idxs[-n_val:]
    img_ids = np.array([i['id'] for i in objects])
    return img_ids[train_idxs], img_ids[val_idxs]


def create_yolo_dataset(
        objects: list, label_mapping: dict, dataset_pth: str,
        val_size: float = 0.1, random_state: int = 42
) -> None:
    """_summary_

    Args:
        objects (list): list of objects. Object is an image with annotation:
        {
            "id": str,
            "path": str,
            "objects": List[dict]: [
                {
                    "cls": int,
                    "xtl": float,
                    "ytl": float,
                    "xbr": float,
                    "ybr": float,
                    "x": float,
                    "y": float
                }
            ]
            
        }.
        label_mapping (dict): labels mapping, e.g. {0: desk, 1: chair}.
        base_imgs_pth (str): _description_
        dataset_pth (str): _description_
        val_size (float, optional): _description_. Defaults to 0.1.
        random_state (int, optional): _description_. Defaults to 42.
    """
    imgs_pth = os.path.join(dataset_pth, "images")
    labels_pth = os.path.join(dataset_pth, "labels")

    shutil.rmtree(dataset_pth, ignore_errors=True)
    os.makedirs(dataset_pth)
    os.makedirs(imgs_pth)
    os.makedirs(os.path.join(imgs_pth, "train"))
    os.makedirs(os.path.join(imgs_pth, "val"))
    os.makedirs(labels_pth)
    os.makedirs(os.path.join(labels_pth, "train"))
    os.makedirs(os.path.join(labels_pth, "val"))

    train_ids, val_ids = split(objects, val_size, random_state)
    train_objects = list(filter(lambda x: x["id"] in train_ids, objects))
    val_objects = list(filter(lambda x: x["id"] in val_ids, objects))

    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "train")] * len(train_ids), train_objects), desc="train labels")
    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "val")] * len(val_ids), val_objects), desc="val labels")

    thread_map(
        cp_img,
        zip([os.path.join(imgs_pth, "train")] * len(train_ids), train_objects),
        desc="train images", max_workers=4
    )
    thread_map(
        cp_img,
        zip([os.path.join(imgs_pth, "val")] * len(val_ids), val_objects),
        desc="val images", max_workers=4
    )

    create_project_file(os.path.join(dataset_pth, "project.yaml"), dataset_pth, label_mapping)

In [87]:
# Function to parse the XML file and merge bounding boxes and points with the same ID
def parse_and_merge_annotations(xml_file, prefix=""):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    annotations = {}
    
    for image in root.findall('image'):
        image_id = image.get('id')
        image_name = image.get('name').replace(".tif", ".jpg")
        width = int(image.get('width'))
        height = int(image.get('height'))
        
        annotations[image_name] = {
            'width': width,
            'height': height,
            'objects': []
        }
        
        boxes = {}
        points = {}
        
        for box in image.findall('box'):
            obj_id = box.find("attribute[@name='ID']").text
            if obj_id not in boxes:
                boxes[obj_id] = []
            boxes[obj_id].append(box)
        
        for point in image.findall('points'):
            obj_id = point.find("attribute[@name='ID']").text
            if obj_id not in points:
                points[obj_id] = []
            points[obj_id].append(point)
        
        for obj_id in boxes:
            if obj_id in points:
                for box in boxes[obj_id]:
                    for point in points[obj_id]:
                        annotations[image_name]['objects'].append({
                            'label': box.get('label'),
                            'bounding_box': [
                                float(box.get('xtl')),
                                float(box.get('ytl')),
                                float(box.get('xbr')),
                                float(box.get('ybr'))
                            ],
                            'points': [
                                (float(p.split(',')[0]), float(p.split(',')[1])) for p in point.get('points').split(';')
                            ],
                            'color': (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
                        })
    
    return annotations

In [88]:
def convert_annotation(annotations, img_paths, prefix=''):
    objects = []
    for img_pth in img_paths:
        img_name = os.path.basename(img_pth)
        if img_name in annotations:
            objects.append({
                "id": f'{prefix}{img_name.split(".")[0]}',
                "path": img_pth,
                "objects": [
                    {
                        "cls": 0,
                        "xtl": obj["bounding_box"][0] / annotations[img_name]["width"],
                        "ytl": obj["bounding_box"][1] / annotations[img_name]["height"],
                        "xbr": obj["bounding_box"][2] / annotations[img_name]["width"],
                        "ybr": obj["bounding_box"][3] / annotations[img_name]["height"],
                        "x": obj["points"][0][0] / annotations[img_name]["width"],
                        "y": obj["points"][0][1] / annotations[img_name]["height"]
                    } for i, obj in enumerate(annotations[img_name]["objects"])
                ]
            })
    return objects

In [89]:
img_paths = [glob.glob(os.path.join(i, "images", "*.jpg")) for i in BASE_FOLDERS]

annotations = [parse_and_merge_annotations(os.path.join(i, "annotations.xml")) for i in BASE_FOLDERS]
sum(len(i) for i in img_paths), sum(len(i) for i in annotations)

(191, 191)

In [90]:
objects = []
for i, j, k in zip(annotations, img_paths, ["", "4pplwalking-"]):
    objects += convert_annotation(i, j, k)

In [91]:
create_yolo_dataset(
    objects, {0: "person"},
    os.path.join('/home/kyamkovyi/projects/researches/people_detection_pg2/quality_reduce_comparison', "cam4_poc"),
    val_size=1, random_state=42
)

train labels: 0it [00:00, ?it/s]
val labels: 191it [00:00, 22858.22it/s]
train images: 0it [00:00, ?it/s]
val images: 191it [00:00, 10408.24it/s]
